# INF01047 - Computação Gráfica e Visualização I
**Laboratório 3: Visualização de Dados**
**Aluno: Leonardo Leal Linhares Dias (326011)**

## O Fator Casa na Premier League: A Influência da Torcida na Agressividade e Arbitragem

Neste laboratório, investigamos o impacto da presença de público nos estádios durante partidas de futebol da Premier League. Comparamos duas temporadas distintas:
* **2018/2019 (Com Torcida):** Cenário tradicional com estádios cheios.
* **2020/2021 (Sem Torcida):** Cenário pandêmico com portões fechados.

O objetivo é visualizar como o "fator casa" se desdobra em métricas de **ofensividade** (Gols e Finalizações) e **disciplina/arbitragem** (Faltas cometidas e Cartões recebidos).

In [6]:
# Instalação de dependências
%pip install pandas
%pip install matplotlib
%pip install altair
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.5/797.5 kB 8.9 MB/s eta 0:00:00ta 0:00:01
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 8.9 MB/s eta 0:00:00ta 0:00:01
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 5.0 MB/s eta 0:00:00
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached 

In [7]:
# Importação das bibliotecas essenciais
import pandas as pd
import altair as alt

# Desabilitando o limite de linhas do Altair para garantir renderização de todos os dados
alt.data_transformers.disable_max_rows()

# 1. Carregamento dos dados
df_1819 = pd.read_csv("season-1819.csv")
df_2021 = pd.read_csv("season-2021.csv")

# 2. Adição da variável categórica de contexto
df_1819['Temporada'] = '18/19 (Com Torcida)'
df_2021['Temporada'] = '20/21 (Sem Torcida)'

# 3. Concatenação dos datasets em um DataFrame unificado
df = pd.concat([df_1819, df_2021], ignore_index=True)

print(f"Total de partidas carregadas: {df.shape[0]}")

Total de partidas carregadas: 760


### 1. Análise de Agressividade e Disciplina (Gráficos de Barras)

Para comparar o desempenho médio global entre mandantes e visitantes, utilizamos um painel de **gráficos de barras agrupadas**. 
* **Marcas:** Linhas (barras).
* **Canais:** Comprimento/Posição vertical (quantitativo) e Cor (categórico - Mandante vs. Visitante).

Esta escolha permite uma comparação direta de magnitudes. Os rótulos de dados foram adicionados no topo das barras para precisão, e a paleta de cores foi escolhida para garantir contraste adequado.

In [ ]:
# 1. Reestruturação (Melt) dos dados
records = []
for idx, row in df.iterrows():
    temp = row['Temporada']
    records.append({'Temporada': temp, 'Mando': 'Casa', 'Métrica': 'Gols', 'Valor': row['FTHG']})
    records.append({'Temporada': temp, 'Mando': 'Fora', 'Métrica': 'Gols', 'Valor': row['FTAG']})
    records.append({'Temporada': temp, 'Mando': 'Casa', 'Métrica': 'Finalizações', 'Valor': row['HS']})
    records.append({'Temporada': temp, 'Mando': 'Fora', 'Métrica': 'Finalizações', 'Valor': row['AS']})
    records.append({'Temporada': temp, 'Mando': 'Casa', 'Métrica': 'Faltas Cometidas', 'Valor': row['HF']})
    records.append({'Temporada': temp, 'Mando': 'Fora', 'Métrica': 'Faltas Cometidas', 'Valor': row['AF']})
    records.append({'Temporada': temp, 'Mando': 'Casa', 'Métrica': 'Cartões Amarelos', 'Valor': row['HY']})
    records.append({'Temporada': temp, 'Mando': 'Fora', 'Métrica': 'Cartões Amarelos', 'Valor': row['AY']})

df_melted = pd.DataFrame(records)

# 2. Agrupando para calcular a média de cada combinação
df_agrupado = df_melted.groupby(['Temporada', 'Métrica', 'Mando'])['Valor'].mean().reset_index()

# 3. Função usando xOffset (método nativo para barras agrupadas que não quebra o layout)
def criar_grafico_metrica(metrica_nome, mostrar_legenda=False):
    df_filtrado = df_agrupado[df_agrupado['Métrica'] == metrica_nome]

    # A legenda aparecerá no topo apenas no gráfico que solicitar
    legenda = alt.Legend(title='Mando de Campo', orient='top') if mostrar_legenda else None

    # Camada 1: Gráfico de barras usando xOffset
    barras = alt.Chart(df_filtrado).mark_bar().encode(
        x=alt.X('Temporada:N', title=None, axis=alt.Axis(labelAngle=0, labelFontSize=12)),
        xOffset='Mando:N',  # Coloca as barras lado a lado
        y=alt.Y('Valor:Q', title=None),
        color=alt.Color('Mando:N', legend=legenda, scale=alt.Scale(domain=['Casa', 'Fora'], range=['#1f77b4', '#ff7f0e'])),
        tooltip=[alt.Tooltip('Mando:N', title='Mando'), alt.Tooltip('Valor:Q', title='Média', format='.2f')]
    )

    # Camada 2: Rótulos de texto acima das barras
    textos = alt.Chart(df_filtrado).mark_text(align='center', baseline='bottom', dy=-3, fontSize=12).encode(
        x=alt.X('Temporada:N'),
        xOffset='Mando:N',
        y=alt.Y('Valor:Q'),
        text=alt.Text('Valor:Q', format='.2f'),
        color=alt.value('black') # Força a cor preta para não mesclar com a cor das barras
    )

    return (barras + textos).properties(
        width=250, height=200, title=alt.TitleParams(f'Média de {metrica_nome}', fontSize=14)
    )

# 4. Criando os 4 gráficos independentes
g1 = criar_grafico_metrica('Gols', mostrar_legenda=True)
g2 = criar_grafico_metrica('Finalizações')
g3 = criar_grafico_metrica('Faltas Cometidas')
g4 = criar_grafico_metrica('Cartões Amarelos')

# 5. Organizando em um Grid 2x2
painel_barras = alt.vconcat(
    alt.hconcat(g1, g2),
    alt.hconcat(g3, g4)
).properties(
    title=alt.TitleParams('Fator Casa na Premier League', fontSize=20, anchor='middle', dy=-15)
).configure_view(
    stroke='transparent'
)

# Exibe o painel
painel_barras

alt.VConcatChart(...)

### 2. A Convergência Ofensiva: O Desempenho Granular dos Clubes

Para entender se o fenômeno observado na média geral se aplica à maioria dos clubes individualmente, utilizamos **Gráficos de Dispersão (Scatter Plots)**.
* **Marcas:** Pontos (representando cada clube).
* **Canais:** Posição horizontal (Média de Gols em Casa) e Posição vertical (Média de Gols Fora).

Adicionamos uma **linha de referência (bissetriz $y = x$)**. Times abaixo da linha marcam mais em casa; times acima marcam mais fora. Esperamos visualizar uma convergência (pontos se aglomerando em torno da linha) na temporada sem torcida, comprovando a neutralização da vantagem de campo.

In [ ]:
# 1. Calculando as médias de gols em casa e fora de cada time
home_avg = df.groupby(['Temporada', 'HomeTeam'])['FTHG'].mean().reset_index()
home_avg.columns = ['Temporada', 'Time', 'Media_Gols_Casa']

away_avg = df.groupby(['Temporada', 'AwayTeam'])['FTAG'].mean().reset_index()
away_avg.columns = ['Temporada', 'Time', 'Media_Gols_Fora']

df_times = pd.merge(home_avg, away_avg, on=['Temporada', 'Time'])

# 2. Configurando as seleções vinculadas pelo mouse (Hover)
# Seleção para a cor/opacidade: se o mouse sai, assume que tudo volta ao normal (empty=True)
hover_cor = alt.selection_point(
    name='hover_cor', on='mouseover', fields=['Time'], clear='mouseout', empty=True
)

# Seleção para o tamanho: se o mouse sai, assume que nada está selecionado (empty=False)
hover_tamanho = alt.selection_point(
    name='hover_tamanho', on='mouseover', fields=['Time'], clear='mouseout', empty=False
)

# 3. Base do scatter plot interativo
base_scatter = alt.Chart(df_times).mark_circle(stroke='black').encode(
    x=alt.X('Media_Gols_Casa:Q', title='Média de Gols em Casa', scale=alt.Scale(domain=[0, 3.5])),
    y=alt.Y('Media_Gols_Fora:Q', title='Média de Gols Fora', scale=alt.Scale(domain=[0, 3.5])),

    # Aumenta o raio apenas do ponto tocado; os demais ficam em 100
    size=alt.condition(hover_tamanho, alt.value(350), alt.value(100)),

    # Destaca as bordas do ponto tocado para um acabamento visual mais refinado
    strokeWidth=alt.condition(hover_tamanho, alt.value(2), alt.value(0.5)),

    # Opacidade normal é 0.9; esmaece (0.2) todos os pontos que não são o selecionado
    opacity=alt.condition(hover_cor, alt.value(0.9), alt.value(0.2)),

    # O Tooltip agora lerá diretamente do elemento gráfico, resolvendo os "undefined"
    tooltip=[
        alt.Tooltip('Time:N', title='Clube'),
        alt.Tooltip('Temporada:N', title='Contexto'),
        alt.Tooltip('Media_Gols_Casa:Q', title='Média Casa', format='.2f'),
        alt.Tooltip('Media_Gols_Fora:Q', title='Média Fora', format='.2f')
    ]
).add_params(
    hover_cor, hover_tamanho
)

# 4. Construção da Linha de Referência Diagonal (Bissetriz y = x)
linha_ref = alt.Chart(pd.DataFrame({'x': [0, 3.5], 'y': [0, 3.5]})).mark_line(
    color='red', strokeDash=[5, 5], opacity=0.5
).encode(
    x='x:Q',
    y='y:Q'
)

# 5. Criando as duas camadas individuais filtradas por temporada
chart_1819 = alt.layer(
    base_scatter.transform_filter(
        alt.datum.Temporada == '18/19 (Com Torcida)'
    ).encode(
        # Cor condicional: Azul para o ponto tocado, cinza para os demais
        color=alt.condition(hover_cor, alt.value('#1f77b4'), alt.value('lightgray'))
    ),
    linha_ref
).properties(
    title=alt.TitleParams('18/19 (Com Torcida)', fontSize=14, fontWeight='bold'),
    width=350, height=350
)

chart_2021 = alt.layer(
    base_scatter.transform_filter(
        alt.datum.Temporada == '20/21 (Sem Torcida)'
    ).encode(
        # Cor condicional: Laranja para o ponto tocado, cinza para os demais
        color=alt.condition(hover_cor, alt.value('#ff7f0e'), alt.value('lightgray'))
    ),
    linha_ref
).properties(
    title=alt.TitleParams('20/21 (Sem Torcida)', fontSize=14, fontWeight='bold'),
    width=350, height=350
)

# 6. Unindo os dois painéis lado a lado com hconcat
grafico_interativo = alt.hconcat(chart_1819, chart_2021).properties(
    title=alt.TitleParams('Dispersão: Ataque em Casa vs. Fora', fontSize=20, anchor='middle', dy=-15)
).configure_view(
    stroke='transparent' # Remove as bordas dos sub-plots
)

# Renderiza a visualização final
grafico_interativo

/home/leonardo/faculdade/2026_1/comp_grafica/lab_visualizacao/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.HConcatChart(...)